In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
base_dir = '/kaggle/input/titanic'
train_path = f'{base_dir}/train.csv'
test_path = f'{base_dir}/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [ ]:
train_df.info()

In [ ]:
test_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.head()

In [ ]:
categorical_cols = ['Sex', 'Embarked', 'Ticket']

for col in categorical_cols:
    print(f'{col}: {train_df[col].unique()}')

# too many unique values in Ticket column. Remove it for the sake of simplicity.

In [ ]:
# there's no duplicated passengerid
train_df.groupby(['PassengerId']).filter(lambda x: len(x) > 1).head()

In [ ]:
train_df.drop(columns=['Cabin', 'Ticket', 'Name'], inplace=True)
test_df.drop(columns=['Cabin', 'Ticket', 'Name'], inplace=True)
train_df.dropna(axis='index', inplace=True)
test_df.dropna(axis='index', inplace=True)

In [ ]:
train_df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_set_df, valid_set_df = train_test_split(train_df, test_size = 0.5)
train_set_df.reset_index(inplace=True)
valid_set_df.reset_index(inplace=True)

In [ ]:
# encode categorical features
from sklearn.preprocessing import OneHotEncoder

def one_hot_encode_util_fit(train_set_df, categorical_cols):
    oh_enc = OneHotEncoder()
    oh_enc_model = oh_enc.fit(train_set_df[categorical_cols])
    oh_col_result_names = []
    for arr in oh_enc_model.categories_:
        for x in arr:
            oh_col_result_names.append('is_' + str(x))

    return oh_enc_model, oh_col_result_names

def one_hot_encode_util_transform(data_set, oh_enc_model, categorical_cols, oh_col_result_names):
    oh_enc_data_set_df = pd.DataFrame(oh_enc_model.transform(data_set[categorical_cols]).toarray())
    oh_enc_data_set_df.columns = oh_col_result_names
    unified_data_set = data_set.join(oh_enc_data_set_df)
    
    return unified_data_set

In [ ]:
categorical_cols = ['Sex', 'Embarked']

oh_enc_model, oh_col_result_names = one_hot_encode_util_fit(train_set_df, categorical_cols)
train_set_df_oh = one_hot_encode_util_transform(train_set_df, oh_enc_model, categorical_cols, oh_col_result_names)
valid_set_df_oh = one_hot_encode_util_transform(valid_set_df, oh_enc_model, categorical_cols, oh_col_result_names)

train_set_df_oh.head(20)

In [ ]:
class_col = 'Survived'
id_col = 'PassengerId'
replaced_onehot_cols = ['Sex', 'Embarked', 'is_male', 'index']
feature_cols = [x for x in train_set_df_oh.columns if x not in [class_col, id_col] + replaced_onehot_cols]

X_train = train_set_df_oh[feature_cols]
y_train = train_set_df_oh[class_col]

X_valid = valid_set_df_oh[feature_cols]
y_valid = valid_set_df_oh[class_col]

In [ ]:
X_train.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=256, max_depth=4, random_state=42)
model = clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_train = model.predict_proba(X_train)[:,1]
y_pred_valid = model.predict_proba(X_valid)[:,1]

print(f'Train AUC: {roc_auc_score(y_train, y_pred_train)}')
print(f'Valid AUC: {roc_auc_score(y_valid, y_pred_valid)}')

In [ ]:
fi_list = []
for name, val in zip(model.feature_importances_, feature_cols):
    fi_list.append((name, val))
    
fi_list.sort(key=lambda tup: tup[0], reverse=True)
fi_list

In [ ]:
from sklearn.inspection import plot_partial_dependence
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (20,15)

plot_partial_dependence(model, X_train, feature_cols, grid_resolution=20)

fig = plt.gcf()
#fig.subplots_adjust(wspace=0.9, hspace=0.9)
fig.tight_layout()


In [ ]:
train_set_df.groupby(['Sex'])['Survived'].sum() / train_set_df.groupby(['Sex'])['Survived'].count()

# clearly, in the ground truth, females has survival rate much higher than males
# why the model predicts females and males have approx. same survival rate ?

In [ ]:
train_set_df_oh.groupby(['is_female'])['Survived'].sum() / train_set_df_oh.groupby(['is_female'])['Survived'].count()

In [ ]:
train_set_df_oh.groupby(['Sex'])['Survived'].sum() / train_set_df_oh.groupby(['Sex'])['Survived'].count()

In [ ]:
train_set_df_oh.groupby(['Sex', 'is_female']).count()
'''
something are wrong here, both males and females have values is_female as 1 AND 0
we should check OneHotEncode process
Mistakes detected: 
    I have kept the index of original dataframe as-is, which is shuffled after train_test_split
    but the one-hot-feature-df has completely new index, which starts from 0
    and I've tried to join the dfs, which resulted in a complete mess with mis-aligned index everywhere
    I;ve fixed it and the model performance has become much better, from valid AUC 0.21 to 0.86
    And is_female has become the most important feature of all
'''

In [ ]:
train_set_df_oh[(train_set_df_oh['Sex'] == 'female') & (train_set_df_oh['is_female'] == 0)]